# 2Dify AI — Free GPU/Hosted ID-V2V Worker

This notebook is the temporary GPU bridge for the 2Dify AI website. It uses the hosted Hugging Face ID-V2V Space when available, so the phone does not need to run the model locally.

**Important:** ID-V2V is a genuine generative video-to-video restylization model, not a color filter. The official model expects a source video, a stylized first frame, and a prompt. The official repository documents generation in 81-frame overlapping clips and a 720p generation path.


In [ ]:
!pip -q install -U gradio_client huggingface_hub imageio-ffmpeg
from pathlib import Path
import shutil, subprocess, json, os
from gradio_client import Client, handle_file
print('Ready')


## 1. Upload source video and the 2D style keyframe
The first frame is the visual specification for the final 2D world. For the strictest 2D result, make this frame completely 2D: character, hair, clothes, objects, buildings, background, shadows and lighting all illustrated. Avoid realistic/3D elements.


In [ ]:
from google.colab import files
print('Upload the source video first')
up_video = files.upload()
source_video = next(iter(up_video))
print('Upload the fully-2D stylized first frame (PNG/JPG)')
up_frame = files.upload()
style_frame = next(iter(up_frame))
print(source_video, style_frame)


## 2. Strict full-2D prompt


In [ ]:
STYLE_PROMPT = '''Fully 2D illustrated animation. Convert every visible element into coherent 2D artwork: characters, faces, eyes, hair, clothes, hands, animals, vehicles, objects, buildings, plants, sky, ground, shadows, highlights, reflections and background. Use flat illustrated shapes, clean 2D linework and hand-drawn/cartoon/anime rendering. Preserve the source video's identity, poses, expressions, gaze, body movement, interactions, lip-sync and camera timing. No photorealism, no live-action texture, no 3D geometry, no CGI look, no physically based materials, no volumetric lighting, no realistic reflections, no photographic skin or hair. The entire final frame must read as a 2D animated scene.'''
print(STYLE_PROMPT)


## 3. Connect to the hosted ID-V2V Space
The official Hugging Face model card lists `hugging-apps/id-v2v` as the hosted Space. The Space may have a queue or availability limits. This notebook does not pretend that the free Space is an unlimited permanent server.


In [ ]:
SPACE = 'hugging-apps/id-v2v'
client = Client(SPACE)
print(client.view_api(all_endpoints=True, print_info=False))


## 4. Find the Space endpoint and run
Run the previous cell first. Gradio exposes the exact endpoint schema used by the current Space. Because the hosted Space can change its UI/API, the notebook intentionally does **not** hard-code an endpoint name that could silently break later.


In [ ]:
# Print only the endpoint names so you can see the current API surface.
api_info = client.view_api(all_endpoints=True, print_info=False)
print(api_info)


### If the Space exposes the standard ID-V2V inference endpoint
Use the exact argument order shown by `view_api()`. The conceptual inputs are: source video, stylized first frame, prompt, and any required model/settings fields. Do not guess argument order if the Space UI has changed.


In [ ]:
# Example template — fill the exact endpoint name and arguments shown by view_api().
# result = client.predict(
#     handle_file(source_video),
#     handle_file(style_frame),
#     STYLE_PROMPT,
#     api_name='/YOUR_CURRENT_ENDPOINT'
# )
# print(result)


## 5. Optional: save/mux original audio
Generative video output may not contain the original audio. If the returned result is a video file, this cell can mux the source audio back into the generated video without re-encoding the video stream when compatible.


In [ ]:
# Set GENERATED to the returned generated-video path before running this cell.
GENERATED = ''
if GENERATED and os.path.exists(GENERATED):
    final = '/content/2dify_full_2d_with_audio.mp4'
    subprocess.run(['ffmpeg','-y','-i',GENERATED,'-i',source_video,'-map','0:v:0','-map','1:a?','-c:v','copy','-c:a','aac','-shortest',final], check=True)
    print(final)
else:
    print('Set GENERATED to the generated video path first.')
